# Coronary Segmentation — Kaggle BUILD notebook

nnU-Net **teacher** → distill **TinyU-Net student** → export ONNX + state_dict, on a Kaggle GPU. Same pipeline as the Colab version, Kaggle-native paths.

### Kaggle setup
1. **Settings → Accelerator → GPU**; **Internet → ON**.
2. **Add Input → Datasets**: attach **ARCADE** (contains `syntax/`) and **DCA1** (the .pgm files). They mount under `/kaggle/input/`.
3. Edit `ARCADE_INPUT` / `DCA1_INPUT` to the real paths, keep `DRY_RUN = True`, **Run All**.

Note: nnU-Net state lives under `/kaggle/working` (persists within a session; **Save Version** to keep it). Kaggle sessions are time-boxed — keep `DRY_RUN` on first.

## 1 · GPU + code + install

In [1]:
import os, sys, torch, glob
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU.'
TORCH = torch.__version__.split('+')[0]     # pin Kaggle's torch so pip can't swap/break it
                                            # (prevents 'duplicate registrations for aten.linspace')
GITHUB_TOKEN = ''                                             # '' if PUBLIC else a PAT
REPO_SLUG = 'jugalmodi0111/interventional-imaging-pipeline'
REPO = '/kaggle/working/interventional-imaging-pipeline'
if not os.path.exists(REPO):
    auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
    !git clone https://{auth}github.com/{REPO_SLUG}.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install nnunetv2 monai SimpleITK onnx onnxscript onnxruntime \
    scikit-image scikit-learn pycocotools pyyaml tqdm "torch=={TORCH}" "numpy<2.1"
from src import env
E = env.setup()                                              # sets nnU-Net env vars under /kaggle/working
print('build env ready | torch', torch.__version__)

Cloning into '/kaggle/working/interventional-imaging-pipeline'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 129 (delta 43), reused 105 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 90.39 KiB | 10.04 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/kaggle/working/interventional-imaging-pipeline
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 88.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━

## 2 · Attach data → symlink into data/raw + DRY_RUN

In [2]:
import glob, os
# auto-discover ARCADE root (contains syntax/) and the DCA1 .pgm folder, wherever Kaggle nested them
sj = glob.glob('/kaggle/input/**/syntax/*/annotations/*.json', recursive=True)
pg = glob.glob('/kaggle/input/**/*.pgm', recursive=True)
assert sj, 'No ARCADE syntax json under /kaggle/input — attach the ARCADE dataset.'
assert pg, 'No DCA1 .pgm under /kaggle/input — attach the DCA1 dataset.'
ARCADE_INPUT = sj[0].split('/syntax/')[0]          # parent of syntax/
DCA1_INPUT   = os.path.dirname(pg[0])              # folder holding the .pgm pairs
DRY_RUN = False
print('ARCADE_INPUT =', ARCADE_INPUT); print('DCA1_INPUT   =', DCA1_INPUT)
os.makedirs('data/raw', exist_ok=True)
!ln -sfn {ARCADE_INPUT} data/raw/arcade
!ln -sfn {DCA1_INPUT}   data/raw/dca1
assert glob.glob('data/raw/arcade/syntax/**/*.json', recursive=True)
assert glob.glob('data/raw/dca1/**/*.pgm', recursive=True)
print('OK: ARCADE syntax json + DCA1 pgm present')

ARCADE_INPUT = /kaggle/input/datasets/jugalmodipesurr/arcade/arcade/arcade
DCA1_INPUT   = /kaggle/input/datasets/jugalmodipesurr/arcade/dca1/dca1
OK: ARCADE syntax json + DCA1 pgm present


## 3 · Standardize → binary vessel pairs + nnU-Net raw

In [3]:
!python -m src.data_prep.arcade_to_coco --config configs/coronary_seg.yaml
!python -m src.data_prep.dca1_to_nnunet --config configs/coronary_seg.yaml
print('processed pairs:', len(glob.glob('data/processed/coronary/img/*.png')))

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
loading annotations into memory...
Done (t=0.08s)
creating index...
index created!
loading annotations into memory...
Done (t=0.27s)
creating index...
index created!
ARCADE -> data/processed/coronary/{img,msk} and /kaggle/working/intv-img/nnUNet_raw/Dataset001_Coronary : 1500 cases
DCA1 -> data/processed/coronary/{img,msk} and /kaggle/working/intv-img/nnUNet_raw/Dataset001_Coronary : +134 cases (raw total 1134)
processed pairs: 1134


## 4 · nnU-Net teacher (2D) + cache logits

In [4]:
DATASET_ID = '001'
# Kaggle GPU sessions cap at ~12h. 250 epochs won't fit -> use 100 (~4-5h on P100/T4). --c resumes if it drops.
TRAINER = 'nnUNetTrainer_5epochs' if DRY_RUN else 'nnUNetTrainer_100epochs'
CACHE = f"{E['root']}/teacher_cache/coronary"; os.makedirs(CACHE, exist_ok=True)
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity
!nnUNetv2_train {DATASET_ID} 2d 0 -tr {TRAINER} --c || nnUNetv2_train {DATASET_ID} 2d 0 -tr {TRAINER}
# build predict as one f-string line — IPython {VAR} substitution breaks across '\' continuations
pred = (f"nnUNetv2_predict -i {os.environ['nnUNet_raw']}/Dataset{DATASET_ID}_Coronary/imagesTr "
        f"-o {CACHE} -d {DATASET_ID} -c 2d -f 0 -tr {TRAINER} --save_probabilities")
!{pred}
print('teacher npz:', len(glob.glob(f'{CACHE}/*.npz')))

Fingerprint extraction...
Dataset001_Coronary
Using <class 'nnunetv2.imageio.natural_image_reader_writer.NaturalImage2DIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.natural_image_reader_writer.NaturalImage2DIO'> as reader/writer
Extracting dataset fingerprint: 100%|███████| 1134/1134 [00:17<00:00, 66.69it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 12, 'patch_size': (np.int64(512), np.int64(512)), 'median_image_size_in_voxels': array([512., 512.]), 'spaci

## 5 · Distill TinyU-Net student

In [5]:
import yaml
from torch.utils.data import DataLoader
from src.models.seg_student import TinyUNet
from src.models.distill import distill, TeacherCacheDataset
from src.eval.metrics import dice, cldice
cfg = yaml.safe_load(open('configs/coronary_seg.yaml'))
ds = TeacherCacheDataset('data/processed/coronary', CACHE, size=cfg['preprocess']['size'])
loader = DataLoader(ds, batch_size=cfg['train']['batch'], shuffle=True, num_workers=2)
student = TinyUNet(in_ch=1, n_classes=1, base=cfg['student']['base_ch'], depth=cfg['student'].get('depth',4))
def quick_eval(m):
    m.eval()
    with torch.no_grad():
        x,y,_ = ds[0]; p = torch.sigmoid(m(x[None].cuda())).cpu().numpy().squeeze() >= 0.5
    m.train(); g = y.numpy().squeeze() > 0.5
    return f'Dice {dice(p,g):.3f} clDice {cldice(p,g):.3f}'
RUNS = f"{E['runs']}/coronary"; os.makedirs(RUNS, exist_ok=True)
distill(student, loader, epochs=(3 if DRY_RUN else cfg['train']['epochs']), lr=cfg['train']['lr'],
        alpha=cfg['distill']['alpha'], T=cfg['distill']['temperature'],
        eval_fn=quick_eval, ckpt=f'{RUNS}/student.pt')

epoch 1/200  kd_loss 1.2408
epoch 2/200  kd_loss 0.7620
epoch 3/200  kd_loss 0.5015
epoch 4/200  kd_loss 0.3742
epoch 5/200  kd_loss 0.3117
epoch 6/200  kd_loss 0.2820
epoch 7/200  kd_loss 0.2664
epoch 8/200  kd_loss 0.2569
epoch 9/200  kd_loss 0.2511
epoch 10/200  kd_loss 0.2482  |  Dice 0.089 clDice 0.083
epoch 11/200  kd_loss 0.2446
epoch 12/200  kd_loss 0.2421
epoch 13/200  kd_loss 0.2403
epoch 14/200  kd_loss 0.2401
epoch 15/200  kd_loss 0.2374
epoch 16/200  kd_loss 0.2368
epoch 17/200  kd_loss 0.2342
epoch 18/200  kd_loss 0.2334
epoch 19/200  kd_loss 0.2339
epoch 20/200  kd_loss 0.2307  |  Dice 0.069 clDice 0.063
epoch 21/200  kd_loss 0.2301
epoch 22/200  kd_loss 0.2289
epoch 23/200  kd_loss 0.2281
epoch 24/200  kd_loss 0.2263
epoch 25/200  kd_loss 0.2293
epoch 26/200  kd_loss 0.2247
epoch 27/200  kd_loss 0.2244
epoch 28/200  kd_loss 0.2235
epoch 29/200  kd_loss 0.2216
epoch 30/200  kd_loss 0.2189  |  Dice 0.093 clDice 0.087
epoch 31/200  kd_loss 0.2202
epoch 32/200  kd_loss 0.21

TinyUNet(
  (enc): ModuleList(
    (0): Sequential(
      (0): Sequential(
        (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (1): Sequential(
        (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
    )
    (1): Sequential(
      (0): Sequential(
        (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (1): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, 

## 6 · Export ONNX + state_dict (handoff to Mac CoreML)

In [6]:
!python -m src.export.to_onnx --weights {RUNS}/student.pt --out {RUNS}/student.onnx
!python -m src.export.quantize_int8 --onnx {RUNS}/student.onnx
!python -m src.eval.edge_benchmark --model {RUNS}/student.onnx
print('\nDownload from output:', f'{RUNS}/student.pt', '(-> Mac: make export-coreml)')

/kaggle/working/interventional-imaging-pipeline/src/export/to_onnx.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(model, torch.randn(*shape), out, opset_version=17,
W0706 21:43:58.677000 41327 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
[torch.onnx] Obtain model graph for `TinyUNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for 